# OSS 4008 — Hospitals in Canada: Chunk 2 (Folium compatibility fix)
## Halifax reference locations and Earth Engine imagery

**Purpose:** Retrieve known hospital coordinates from OpenStreetMap, display actual Sentinel-2 Earth Engine imagery with reference markers, and export a CSV. This is preparation for image annotation and AI training, **not** a trained model.

**Fix:** Uses Folium directly for mapping to avoid the `geemap` / `BoxKeyError: xyz_to_folium` bug in some Colab library combinations. Earth Engine is still required and used for the satellite imagery. No geemap installation is needed.

Run top-to-bottom in Google Colab. If you already imported geemap and hit the error, choose **Runtime → Restart session**, then run all cells.


In [ ]:
%pip -q install "earthengine-api>=1.5,<2" folium requests pandas


### 1 — Authenticate Earth Engine
Google Colab may ask you to approve the Google account. This is necessary and cannot be automated for you. Uses the same Google Cloud project as your successful first notebook.

In [ ]:
import ee
import folium
import pandas as pd
import requests
from pathlib import Path

PROJECT_ID = "infra-bedrock-411215"
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine server test:", ee.Number(42).getInfo())


### 2 — Geographic area
Start small: Halifax, Nova Scotia. The coordinates use decimal degrees in WGS84 (latitude / longitude).

In [ ]:
CITY = "Halifax"
SOUTH, WEST, NORTH, EAST = 44.55, -63.75, 44.75, -63.45
CENTER = [44.6488, -63.5752]
BOUNDS = ee.Geometry.Rectangle([WEST, SOUTH, EAST, NORTH])
print(f"Area: {CITY} | {SOUTH}, {WEST}, {NORTH}, {EAST}")

### 3 — Optional hospital reference locations from OpenStreetMap
This stage retrieves existing mapped locations (not image-based detections). Overpass is a public service, so its availability can vary. A failure will not prevent the satellite imagery test.


In [ ]:
# Fetch optional hospital reference points. This is NOT needed for the satellite layer.
# Public Overpass servers sometimes time out; continue even if both fail.
query = f"""[out:json][timeout:15];
(
  node["amenity"="hospital"]({SOUTH},{WEST},{NORTH},{EAST});
  way["amenity"="hospital"]({SOUTH},{WEST},{NORTH},{EAST});
  relation["amenity"="hospital"]({SOUTH},{WEST},{NORTH},{EAST});
);
out center;"""

endpoints = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]
elements = []
osm_available = False
for endpoint in endpoints:
    print(f"Checking OpenStreetMap via {endpoint} ...", flush=True)
    try:
        response = requests.post(endpoint, data={"data": query}, timeout=(5, 18))
        response.raise_for_status()
        elements = response.json().get("elements", [])
        osm_available = True
        print(f"OSM features returned: {len(elements)}")
        break
    except (requests.RequestException, ValueError) as exc:
        print(f"Could not retrieve OSM data: {type(exc).__name__}: {str(exc)[:160]}")
if not osm_available:
    print("WARNING: OpenStreetMap references unavailable. Satellite imagery test will continue; hospital markers and CSV will be empty.")


In [ ]:
records = []
seen = set()
for item in elements:
    tags = item.get("tags", {})
    point = item if "lat" in item and "lon" in item else item.get("center", {})
    if not {"lat", "lon"}.issubset(point):
        continue
    key = (item.get("type"), item.get("id"))
    if key in seen:
        continue
    seen.add(key)
    records.append({
        "osm_type": item.get("type"),
        "osm_id": item.get("id"),
        "name": tags.get("name", "Unnamed hospital"),
        "latitude": point["lat"],
        "longitude": point["lon"],
        "amenity": tags.get("amenity", ""),
        "source": "OpenStreetMap (Overpass API)",
    })

columns = ["osm_type", "osm_id", "name", "latitude", "longitude", "amenity", "source"]
hospitals = pd.DataFrame(records, columns=columns)
print(f"Retrieved {len(hospitals)} OSM hospital features (not model predictions)")
if hospitals.empty:
    print("No hospital markers available. The satellite-image layer can still be tested.")
display(hospitals.head(20))

### 4 — Retrieve Sentinel-2 imagery
This uses surface reflectance RGB bands (red, green, blue). The median reduces cloud contamination across multiple dates, but does not guarantee a completely cloud-free map.

In [ ]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(BOUNDS)
      .filterDate("2025-06-01", "2025-10-01")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 35)))
print("Sentinel-2 scenes:", s2.size().getInfo())
image = s2.median().clip(BOUNDS)
visualization = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000, "gamma": 1.15}

### 5 — Interactive map (no `geemap` dependency)
Red markers are OpenStreetMap reference hospital locations, not AI predictions. The RGB satellite layer is generated from Google Earth Engine and rendered through Folium tile URLs.


In [ ]:
# Use Folium directly to avoid the geemap/Box xyz_to_folium compatibility error.
# This is an actual Earth Engine image tile layer; no satellite raster download is needed.
map_info = image.getMapId(visualization)

m = folium.Map(location=CENTER, zoom_start=11, tiles="OpenStreetMap")
folium.TileLayer(
    tiles=map_info["tile_fetcher"].url_format,
    attr="Imagery: Copernicus Sentinel-2 via Google Earth Engine",
    name="Sentinel-2 RGB (summer 2025)",
    overlay=True,
    control=True,
    max_zoom=19,
).add_to(m)

for _, row in hospitals.iterrows():
    folium.Marker(
        location=[float(row["latitude"]), float(row["longitude"])],
        tooltip=str(row["name"]),
        popup=folium.Popup(f"{row['name']}<br>OSM reference (not AI detected)", max_width=280),
        icon=folium.Icon(color="red", icon="plus-sign"),
    ).add_to(m)

folium.LayerControl().add_to(m)
display(m)


### 6 — Export the reference CSV
The CSV is saved temporarily in the Colab session. Use the optional download command to keep a copy. Large satellite images are not downloaded or committed to GitHub.

In [ ]:
output_path = Path("/content/halifax_osm_hospitals.csv")
hospitals.to_csv(output_path, index=False)
print("CSV saved:", output_path)
print("Rows:", len(hospitals))
# Optional: from google.colab import files; files.download(str(output_path))

## Completion check and connection to the project goal
- Earth Engine request returns `42`.
- OSM reference table lists available hospital features.
- Folium map displays an Earth Engine satellite layer with red reference markers.
- CSV is created in the Colab session.

**What this accomplishes:** Finds candidate hospital sites so we know *where to collect and label imagery*. It does **not** teach an AI to recognize a hospital. The next substantive stage is high-resolution imagery and manual bounding-box annotations, followed by model training and testing. Sentinel-2's 10 m RGB pixels are primarily for geographic context; imagery detail must be assessed before building-detection training.

Data: © OpenStreetMap contributors (ODbL), Copernicus Sentinel-2 through Google Earth Engine. Check attribution and licences before publishing.
